In [1]:
import json
import os

# 경로 설정
INPUT_PATH = r"D:\kpop-dance-dtw-analysis\dtw-analysis\data\json\expert_lovedive.json"
OUTPUT_PATH = r"D:\kpop-dance-dtw-analysis\dtw-analysis\data\json\expert_lovedive_visible.json"

# 데이터 로드
with open(INPUT_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

print(f"✅ 데이터 로드 완료")
print(f"총 프레임 수: {len(data['frames'])}")
print(f"FPS: {data['metadata']['fps']}")

✅ 데이터 로드 완료
총 프레임 수: 4369
FPS: 24.0


In [2]:
# 임계값 설정
VISIBILITY_THRESHOLD = 0.5  # 0.5 이상이면 visible: true

def apply_visibility_filter(data, threshold):
    filtered_frames = []

    for frame in data["frames"]:
        new_frame = {
            "frame_index": frame["frame_index"],
            "timestamp_ms": frame["timestamp_ms"],
            "landmarks": []
        }

        if frame["landmarks"] is None:
            new_frame["landmarks"] = None
        else:
            for lm in frame["landmarks"]:
                new_frame["landmarks"].append({
                    "id": lm["id"],
                    "x": lm["x"],
                    "y": lm["y"],
                    "z": lm["z"],
                    "visibility": lm["visibility"],
                    "visible": lm["visibility"] >= threshold  # ✅ True/False 추가
                })

        filtered_frames.append(new_frame)

    return {
        "metadata": {
            **data["metadata"],
            "visibility_threshold": threshold  # 임계값도 메타데이터에 기록
        },
        "frames": filtered_frames
    }

print(f"✅ 함수 정의 완료 (threshold={VISIBILITY_THRESHOLD})")

✅ 함수 정의 완료 (threshold=0.5)


In [3]:
# 필터링 실행
filtered_data = apply_visibility_filter(data, VISIBILITY_THRESHOLD)

# 저장
with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(filtered_data, f, indent=2, ensure_ascii=False)

print(f"✅ 저장 완료 → {OUTPUT_PATH}")

# 결과 샘플 확인
sample = filtered_data["frames"][0]["landmarks"]
print(f"\n📊 첫 번째 프레임 샘플 (관절 0~4):")
for lm in sample[:5]:
    print(f"  id={lm['id']} | visibility={lm['visibility']:.4f} | visible={lm['visible']}")

✅ 저장 완료 → D:\kpop-dance-dtw-analysis\dtw-analysis\data\json\expert_lovedive_visible.json

📊 첫 번째 프레임 샘플 (관절 0~4):
  id=0 | visibility=1.0000 | visible=True
  id=1 | visibility=1.0000 | visible=True
  id=2 | visibility=1.0000 | visible=True
  id=3 | visibility=1.0000 | visible=True
  id=4 | visibility=1.0000 | visible=True


In [4]:
total_landmarks = 0
visible_true = 0
visible_false = 0

for frame in filtered_data["frames"]:
    if frame["landmarks"] is None:
        continue
    for lm in frame["landmarks"]:
        total_landmarks += 1
        if lm["visible"]:
            visible_true += 1
        else:
            visible_false += 1

print(f"📊 전체 관절 수: {total_landmarks:,}")
print(f"  visible=True : {visible_true:,} ({visible_true/total_landmarks*100:.1f}%)")
print(f"  visible=False: {visible_false:,} ({visible_false/total_landmarks*100:.1f}%)")

📊 전체 관절 수: 140,580
  visible=True : 129,947 (92.4%)
  visible=False: 10,633 (7.6%)
